In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * DATABASE OPTIMIZATION — Indexes, Query Tuning & Pagination
 * ──────────────────────────────────────────────────────────────────────
 * 1.  INDEXES (Basic to Advanced)
 *     1.1  What is an Index?
 *     1.2  Where is Index Stored?
 *     1.3  What is B-Tree?
 *     1.4  What Data Does Index Store?
 *     1.5  How Fast is Index Search?
 *     1.6  Table Scan vs Index Scan
 *     1.7  Where to Use Index?
 *     1.8  Composite Indexes
 *     1.9  How to Choose the Right Index?
 *     1.10 Cost of Index
 * 2.  QUERY OPTIMIZATION
 *     2.1  Do Not Use SELECT *
 *     2.2  Sargable Conditions
 *     2.3  What is Caching?
 *     2.4  Use EXPLAIN to Check Query
 * 3.  LIMIT AND BATCHING
 *     3.1  Use LIMIT
 *     3.2  Batching — Process Data in Small Parts
 * 4.  PAGINATION — OFFSET vs CURSOR
 *     4.1  OFFSET Pagination (Slow)
 *     4.2  CURSOR Pagination (Fast)
 * 5.  GOLDEN RULES
 * 6.  PRACTICE EXERCISES
 * 7.  QUICK REFERENCE CARD
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * SAMPLE TABLE — orders (Sab examples isi pe hain)
 * ──────────────────────────────────────────────────────────────────────
 *
 *   Ye ek simple orders table hai jaise Amazon ki order history.
 *   Raj, Priya, Amit, Neha — inke orders store kiye hain.
 *   Saare examples isi table se chalenge.
 *
 * ┌──────────┬─────────┬──────────────┬────────────┬────────────┬─────────┐
 * │ order_id │ user_id │ product_name │   status   │ order_date │  price  │
 * ├──────────┼─────────┼──────────────┼────────────┼────────────┼─────────┤
 * │    1     │    1    │   Laptop     │ DELIVERED  │ 2024-01-15 │ 50000   │
 * │    2     │    2    │   Mouse      │    PAID    │ 2024-01-16 │ 500     │
 * │    3     │    1    │  Keyboard    │ DELIVERED  │ 2024-01-17 │ 1500    │
 * │    4     │    3    │  Monitor     │   PLACED   │ 2024-01-18 │ 10000   │
 * │    5     │    2    │ Headphones   │ CANCELLED  │ 2024-01-19 │ 2000    │
 * │    6     │    1    │  USB Cable   │ DELIVERED  │ 2024-01-20 │ 300     │
 * │    7     │    4    │   Laptop     │    PAID    │ 2024-01-21 │ 55000   │
 * │    8     │    3    │   Mouse      │ DELIVERED  │ 2024-01-22 │ 500     │
 * │    9     │    1    │  Monitor     │   PLACED   │ 2024-01-23 │ 12000   │
 * │   10     │    2    │  Keyboard    │ DELIVERED  │ 2024-01-24 │ 1500    │
 * └──────────┴─────────┴──────────────┴────────────┴────────────┴─────────┘
 *
 * Users:
 *   Raj   (user_id=1) → Laptop, Keyboard, USB Cable, Monitor
 *   Priya (user_id=2) → Mouse, Headphones, Keyboard
 *   Amit  (user_id=3) → Monitor, Mouse
 *   Neha  (user_id=4) → Laptop
 */


CREATE TABLE orders (
    order_id     SERIAL PRIMARY KEY,
    user_id      INTEGER NOT NULL,
    product_name VARCHAR(100),
    status       VARCHAR(20),
    order_date   DATE,
    price        DECIMAL(10,2)
);

INSERT INTO orders (user_id, product_name, status, order_date, price) VALUES
    (1, 'Laptop',     'DELIVERED', '2024-01-15', 50000),
    (2, 'Mouse',      'PAID',      '2024-01-16', 500),
    (1, 'Keyboard',   'DELIVERED', '2024-01-17', 1500),
    (3, 'Monitor',    'PLACED',    '2024-01-18', 10000),
    (2, 'Headphones', 'CANCELLED', '2024-01-19', 2000),
    (1, 'USB Cable',  'DELIVERED', '2024-01-20', 300),
    (4, 'Laptop',     'PAID',      '2024-01-21', 55000),
    (3, 'Mouse',      'DELIVERED', '2024-01-22', 500),
    (1, 'Monitor',    'PLACED',    '2024-01-23', 12000),
    (2, 'Keyboard',   'DELIVERED', '2024-01-24', 1500);

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-1: INDEXES — Basic to Advanced
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * 1.1 WHAT IS AN INDEX?
 * ──────────────────────
 *   Index ek phonebook ya dictionary ki tarah hota hai.
 *   Database ko har row check nahi karna padta — seedha matching row pe jaata hai.
 *
 * Real-life example: School mein roll number dhundna
 *
 *   WITHOUT INDEX (Slow way):
 *     Teacher sab students se poochh rahi hai: "Tumhara roll number kya hai?"
 *     Student 1: "1" → Nahi
 *     Student 2: "2" → Nahi
 *     ... 50 tak
 *     Total checks: 50 | Time: 5 minutes
 *
 *   WITH INDEX (Fast way):
 *     Teacher ek register dekhti hai (index):
 *     "Roll number 50 → 3rd row, 2nd bench"
 *     Seedha wahan jaati hai.
 *     Total checks: 1 | Time: 10 seconds
 *
 * Same DATABASE mein:
 */


-- WITHOUT INDEX: Database sab 10 rows check karta hai
SELECT * FROM orders WHERE user_id = 1;

-- WITH INDEX: Database seedha matching rows pe jaata hai
CREATE INDEX idx_orders_user_id ON orders (user_id);
SELECT * FROM orders WHERE user_id = 1;

-- Index hatana
DROP INDEX idx_orders_user_id;

In [ ]:
/**
 * 1.2 WHERE IS INDEX STORED? — Index ka Storage Location
 * ────────────────────────────────────────────────────────
 *   Index DISK pe store hota hai — bilkul table data ki tarah.
 *   Lekin index table se bahut chhota hota hai.
 *   Frequently used index pages RAM mein cache bhi ho jaate hain.
 *
 * ┌─────────────────────────────────────────────────────────────────────┐
 * │                         YOUR DATABASE                               │
 * ├─────────────────────────────────────────────────────────────────────┤
 * │                                                                     │
 * │   HARD DISK (Permanent storage)                                     │
 * │   ┌──────────────────────────┐  ┌──────────────────────────────┐    │
 * │   │     TABLE DATA           │  │        INDEX DATA            │    │
 * │   │  (Actual orders table)   │  │      (Search helper)         │    │
 * │   │  - All 10 rows           │  │  - Sorted by user_id         │    │
 * │   │  - All 6 columns         │  │  - Only user_id + pointer    │    │
 * │   │  - Size: 10 KB           │  │  - Size: 2 KB                │    │
 * │   └──────────────────────────┘  └──────────────────────────────┘    │
 * │                                                                     │
 * │   MEMORY / RAM (Fast cache)                                         │
 * │   ┌──────────────────────────────────────────────────────────────┐  │
 * │   │  Frequently used index pages automatically cache hote hain   │  │
 * │   │  for even faster access                                      │  │
 * │   └──────────────────────────────────────────────────────────────┘  │
 * └─────────────────────────────────────────────────────────────────────┘
 *
 * KEY POINTS:
 *   1. Index DISK pe store hota hai (permanent)
 *   2. Index extra space leta hai (kabhi kabhi table se bhi zyada)
 *   3. Frequently used index parts RAM mein cache ho jaate hain
 *   4. Har index disk pe ek alag file hai
 */


-- Index kitna space le raha hai check karo
SELECT
    tablename,
    indexname,
    pg_size_pretty(pg_indexes_size(tablename::regclass)) AS total_index_size
FROM pg_indexes
WHERE tablename = 'orders';

-- Table size vs Index size compare karo
SELECT
    pg_size_pretty(pg_relation_size('orders'))  AS table_size,
    pg_size_pretty(pg_indexes_size('orders'))   AS total_index_size;

In [ ]:
/**
 * 1.3 WHAT IS B-TREE? — Index Internally Kaise Kaam Karta Hai
 * ──────────────────────────────────────────────────────────────
 *   B-Tree wo technology hai jo index ko fast banati hai.
 *   Data ko ek tree structure mein organize karta hai — jaise family tree.
 *   Har step search area HALF kar deta hai.
 *
 * Real-life example: Dictionary mein "Monitor" dhundna
 *   Step 1: Middle page (500) kholo
 *   Step 2: "Monitor" pehle aata hai → left half (1-500) jao
 *   Step 3: Middle of left half (250) kholo
 *   Step 4: Tab tak repeat karo jab tak word mile
 *   Har step mein search area half ho jaata hai!
 *
 * Number example: 1-100 mein se 75 dhundho
 *   Step 1: 50 check karo → 75 bada hai → right jao
 *   Step 2: 75 mil gaya!
 *   Sirf 2 steps — 75 ki jagah!
 *
 * B-TREE VISUAL on user_id column:
 *   user_id values: 1,1,1,1,2,2,2,3,3,4
 *
 *                              ┌─────────┐
 *                              │    2    │  ← ROOT NODE (Top level)
 *                              │ (Middle)│    (Yahan se shuru hota hai)
 *                              └────┬────┘
 *                                   │
 *              ┌────────────────────┼────────────────────┐
 *              ▼                    ▼                    ▼
 *         ┌─────────┐          ┌─────────┐          ┌─────────┐
 *         │    1    │          │    2    │          │   3,4   │
 *         │ (Left)  │          │ (Middle)│          │ (Right) │
 *         └────┬────┘          └────┬────┘          └────┬────┘
 *              │                    │                    │
 *        ┌─────┼─────┐         ┌─────┼─────┐         ┌─────┼─────┐
 *        ▼     ▼     ▼         ▼     ▼     ▼         ▼     ▼     ▼
 *     ┌───┐ ┌───┐ ┌───┐     ┌───┐ ┌───┐ ┌───┐     ┌───┐ ┌───┐ ┌───┐
 *     │ 1 │ │ 1 │ │ 1 │     │ 2 │ │ 2 │ │ 2 │     │ 3 │ │ 3 │ │ 4 │
 *     │Row│ │Row│ │Row│     │Row│ │Row│ │Row│     │Row│ │Row│ │Row│
 *     │ 1 │ │ 3 │ │ 6 │     │ 2 │ │ 5 │ │10 │     │ 4 │ │ 8 │ │ 7 │
 *     └───┘ └───┘ └───┘     └───┘ └───┘ └───┘     └───┘ └───┘ └───┘
 *    (Leaf nodes — actual row pointers)
 *
 * user_id=2 dhundha toh kya hoga:
 *   Step 1: ROOT pe jao (value 2)
 *   Step 2: user_id=2 match kiya → middle branch lo
 *   Step 3: Leaf nodes mein user_id=2 dhundho
 *   Step 4: Row pointers milo (rows 2, 5, 10)
 *   Step 5: Seedha un rows pe jao
 *   Total: Sirf 3-4 steps!
 *
 * SPEED:
 *   1,000,000 records ke liye:
 *     Without B-Tree: 1,000,000 checks
 *     With B-Tree:    sirf ~20 checks   (log₂ of 1,000,000 ≈ 20)
 *     2^20 = 1,048,576 → 20 steps mein kuch bhi dhundo!
 *     → 50,000 times faster!
 */

In [ ]:
/**
 * 1.4 WHAT DATA DOES INDEX STORE?
 * ──────────────────────────────────
 *   Index sirf DO cheezein store karta hai:
 *   1. Column value (key) — jis column pe index hai uski value
 *   2. Row pointer (CTID) — actual row kahan hai (block_number, row_offset)
 *
 *   Index STORE NAHI KARTA:
 *   → Doosre columns (product_name, order_date, price, etc.)
 *   → Ye sirf actual table mein rehte hain
 *
 * INDEX (idx_user_id) ka structure:
 * ┌─────────────────────────────────────────────────────────────────────┐
 * │                    INDEX (idx_user_id)                              │
 * ├─────────────────────────────────────────────────────────────────────┤
 * │  Sorted by user_id:                                                 │
 * │  ┌─────────────┬──────────────────────────┐                         │
 * │  │  user_id    │  Pointer to row          │                         │
 * │  │  (Key)      │  (Row kahan hai)         │                         │
 * │  ├─────────────┼──────────────────────────┤                         │
 * │  │     1       │  → Row 1  (order_id = 1) │                         │
 * │  │     1       │  → Row 3  (order_id = 3) │                         │
 * │  │     1       │  → Row 6  (order_id = 6) │                         │
 * │  │     1       │  → Row 9  (order_id = 9) │                         │
 * │  │     2       │  → Row 2  (order_id = 2) │                         │
 * │  │     2       │  → Row 5  (order_id = 5) │                         │
 * │  │     2       │  → Row 10 (order_id = 10)│                         │
 * │  │     3       │  → Row 4  (order_id = 4) │                         │
 * │  │     3       │  → Row 8  (order_id = 8) │                         │
 * │  │     4       │  → Row 7  (order_id = 7) │                         │
 * │  └─────────────┴──────────────────────────┘                         │
 * │  TOTAL SIZE: Sirf 10 entries × 2 values = Bahut chhota!             │
 * └─────────────────────────────────────────────────────────────────────┘
 *
 * COVERING INDEX — Special case:
 *   Agar index mein query ke saare columns hain toh database ko
 *   actual table hi nahi padhni padti!
 */


-- Covering index banao
CREATE INDEX idx_covering ON orders (user_id, order_id, status);

-- Ye query sirf index use karegi (table access nahi)
SELECT user_id, order_id, status FROM orders WHERE user_id = 1;

-- Check karo "Index Only Scan" aa raha hai?
EXPLAIN ANALYZE SELECT user_id, order_id, status FROM orders WHERE user_id = 1;

In [ ]:
/**
 * 1.5 HOW FAST IS INDEX SEARCH? — Speed Comparison
 * ───────────────────────────────────────────────────
 *   Chhoti table (10 rows) ke liye shayad farak na lage.
 *   Lekin 10 lakh rows mein index ka asli jadoo dikhta hai.
 *
 * 10-row table mein user_id=1 dhundho:
 *
 *   TABLE SCAN (No index):
 *     Row 1:  user_id=1? YES → lo
 *     Row 2:  user_id=2? NO  → skip
 *     Row 3:  user_id=1? YES → lo
 *     Row 4:  user_id=3? NO  → skip
 *     Row 5:  user_id=2? NO  → skip
 *     Row 6:  user_id=1? YES → lo
 *     Row 7:  user_id=4? NO  → skip
 *     Row 8:  user_id=3? NO  → skip
 *     Row 9:  user_id=1? YES → lo
 *     Row 10: user_id=2? NO  → skip
 *     Total: 10 checks
 *
 *   INDEX SCAN (With index):
 *     Index dekha: user_id=1 → rows 1, 3, 6, 9
 *     Seedha un 4 rows pe gaya
 *     Total: 4 checks only
 *
 * ┌─────────────────────────────────────────────────────────────────────┐
 * │              1,000,000 ROWS KE LIYE SPEED COMPARISON                │
 * ├─────────────────────────────────────────────────────────────────────┤
 * │  TABLE SCAN (No index):                                             │
 * │    → Checks: 1,000,000 rows                                         │
 * │    → Time: 2-3 seconds                                              │
 * │                                                                     │
 * │  INDEX SCAN (With index):                                           │
 * │    → Index mein: ~20 steps (B-Tree)                                 │
 * │    → Matching rows: ~100 rows                                       │
 * │    → Total: ~120 checks                                             │
 * │    → Time: 0.01 seconds                                             │
 * │                                                                     │
 * │  RESULT: Index = 200x FASTER!                                       │
 * └─────────────────────────────────────────────────────────────────────┘
 */


\timing on

-- Without index (slow)
SELECT * FROM orders WHERE user_id = 1;

-- Index banao
CREATE INDEX idx_user_id ON orders (user_id);

-- With index (fast)
SELECT * FROM orders WHERE user_id = 1;

\timing off

In [ ]:
/**
 * 1.6 TABLE SCAN vs INDEX SCAN — Bad vs Good
 * ────────────────────────────────────────────
 *   Table Scan = BURA — poori table scan karta hai
 *   Index Scan = ACHA — seedha matching rows pe jaata hai
 *
 *   EXPLAIN output mein "Seq Scan" dikhna = BAD (no index)
 *   EXPLAIN output mein "Index Scan" dikhna = GOOD
 */


-- TABLE SCAN example (no index on user_id)
EXPLAIN SELECT * FROM orders WHERE user_id = 1;
-- "Seq Scan" dikhega → BAD

-- INDEX SCAN example
CREATE INDEX idx_orders_user_id ON orders (user_id);
EXPLAIN SELECT * FROM orders WHERE user_id = 1;
-- "Index Scan" dikhega → GOOD


/**
 * TABLE SCAN kya karta hai (internally):
 *   Row 1:  user_id=1? YES → lo
 *   Row 2:  user_id=2? NO  → skip
 *   ... 10 lakh rows tak
 *   Sab check kiye! Bahut slow.
 *
 * INDEX SCAN kya karta hai (internally):
 *   Step 1: Index mein user_id=1 dhundho
 *   Step 2: Index bolata hai: rows 1, 3, 6, 9
 *   Step 3: Seedha un 4 rows pe jao
 *   Step 4: Baaki sab skip!
 *   Bahut fast.
 */

In [ ]:
/**
 * 1.7 WHERE TO USE INDEX? — WHERE, JOIN, ORDER BY, GROUP BY
 * ───────────────────────────────────────────────────────────
 * Rule: Index wahan lagao jahan ye columns aate hain:
 * 1. WHERE conditions mein
 * 2. JOIN conditions mein
 * 3. ORDER BY mein
 * 4. GROUP BY mein
 */

In [ ]:
-- SCENARIO 1: WHERE clause
-- Agar user_id se search karte ho → index lagao user_id pe

-- ❌ BAD: No index (slow)
SELECT * FROM orders WHERE user_id = 5;

-- ✅ GOOD: Index ke saath (fast)
CREATE INDEX idx_user_id ON orders (user_id);
SELECT * FROM orders WHERE user_id = 5;

In [ ]:
-- SCENARIO 2: JOIN condition
-- JOIN column pe index lagao

-- users table banao
CREATE TABLE users (user_id SERIAL PRIMARY KEY, name VARCHAR(100));
INSERT INTO users (user_id, name) VALUES (1,'Raj'),(2,'Priya'),(3,'Amit'),(4,'Neha');

-- ❌ BAD: No index (slow join)
SELECT o.order_id, u.name
FROM orders o JOIN users u ON o.user_id = u.user_id;

-- ✅ GOOD: Index ke saath (fast join)
CREATE INDEX idx_orders_user_id ON orders (user_id);
SELECT o.order_id, u.name
FROM orders o JOIN users u ON o.user_id = u.user_id;

In [ ]:
-- SCENARIO 3: ORDER BY clause
-- Sort column pe index lagao

-- ❌ BAD: No index (slow sort)
SELECT * FROM orders ORDER BY order_date;

-- ✅ GOOD: Index ke saath (already sorted)
CREATE INDEX idx_order_date ON orders (order_date);
SELECT * FROM orders ORDER BY order_date;

In [ ]:
-- SCENARIO 4: GROUP BY clause
-- Group column pe index lagao

-- ❌ BAD: No index (slow grouping)
SELECT user_id, COUNT(*) FROM orders GROUP BY user_id;

-- ✅ GOOD: Index ke saath (fast)
CREATE INDEX idx_user_id ON orders (user_id);
SELECT user_id, COUNT(*) FROM orders GROUP BY user_id;

In [ ]:
/**
 * 1.8 COMPOSITE INDEXES — Ek Saath Do Columns Pe Index
 * ──────────────────────────────────────────────────────
 *   Composite Index = 2 ya zyada columns pe ek index.
 *   Jab WHERE clause mein multiple columns filter karo toh use karo.
 *
 *   IMPORTANT: Column order bahut matter karta hai!
 *   Rule: Sabse zyada unique values wala column PEHLE rakho.
 *
 *   Example:
 *     user_id: 1000 unique values → zyada unique
 *     status:  5 unique values    → kam unique
 *     → user_id pehle rakho!
 */


-- Example: user_id aur status dono se search karo
SELECT * FROM orders WHERE user_id = 1 AND status = 'DELIVERED';

-- ✅ Composite index banao
CREATE INDEX idx_user_status ON orders (user_id, status);
SELECT * FROM orders WHERE user_id = 1 AND status = 'DELIVERED';

-- ✅ CORRECT order (more selective column first)
CREATE INDEX idx_user_status_correct ON orders (user_id, status);

-- ❌ WRONG order (slower)
CREATE INDEX idx_status_user_wrong ON orders (status, user_id);


/**
 * Column order ka visual example:
 *
 * idx_user_status (user_id, status):
 * ┌─────────┬──────────────┬────────────────────┐
 * │ user_id │ status       │ Row pointer        │
 * ├─────────┼──────────────┼────────────────────┤
 * │    1    │ DELIVERED    │ → rows 1, 3, 6     │
 * │    1    │ PLACED       │ → row 9            │
 * │    2    │ CANCELLED    │ → row 5            │
 * │    2    │ DELIVERED    │ → row 10           │
 * │    2    │ PAID         │ → row 2            │
 * └─────────┴──────────────┴────────────────────┘
 *
 * user_id=1, status='DELIVERED' dhundho:
 *   Seedha B-Tree mein (1, DELIVERED) dhundho → instant!
 */

In [ ]:
/**
 * 1.9 HOW TO CHOOSE THE RIGHT INDEX?
 * ─────────────────────────────────────
 *   Query dekho aur ussi ke hisaab se index banao.
 *   Equality columns pehle, range columns baad mein, ORDER BY last mein.
*/

/** 
 * Example query:
 */


SELECT order_id, user_id, order_date, price
FROM orders
WHERE user_id = 1
  AND order_date >= '2024-01-01'
  AND order_date <= '2024-01-31'
ORDER BY order_date
LIMIT 10;


/**
 * Konsa index best hai?
 *
 * OPTION 1: Sirf user_id pe index
 *   → user_id=1 wale sab rows milenge
 *   → Phir order_date filter karo
 *   → Phir sort karo
 *   → Zyada rows scan
 *
 * OPTION 2: Composite (user_id, order_date) — BEST ✅
 *   → user_id filter → first column
 *   → order_date filter → second column (already sorted in index)
 *   → ORDER BY → already sorted
 *   → LIMIT 10 → jaldi ruk jaata hai
 *   → Sirf 10 rows scan!
 */


-- Best index for this query:
CREATE INDEX idx_user_orderdate ON orders (user_id, order_date);


/**
 * RULES for index column order:
 *   1. EQUALITY columns pehle  → WHERE user_id = 1
 *   2. RANGE columns baad mein → WHERE order_date > '2024-01-01'
 *   3. ORDER BY columns last   → agar pehle covered na hua ho
 */

In [ ]:
/**
 * 1.10 COST OF INDEX — Indexes Free Nahi Hain
 * ─────────────────────────────────────────────
 *   Index SELECT fast karta hai — lekin INSERT/UPDATE SLOW karta hai.
 *   Har ek index ek extra write hai jab bhi data change hota hai.
 *
 * COST 1: Storage Space
 *   → Indexes extra disk space lete hain.
 *
 * COST 2: Slower Writes
 *   → Table mein 3 indexes hain toh:
 *     1 INSERT = Table mein write + Index 1 update + Index 2 update + Index 3 update
 *     = 4 writes for 1 insert!
 *
 * INSERT ke time kya hota hai internally:
 *   INSERT INTO orders VALUES (11, 2, 'Speaker', 'PLACED', '2024-01-25', 3000);
 *
 *   Step 1: Table mein row add karo
 *   Step 2: user_id index mein user_id=2 add karo
 *   Step 3: Index mein sahi position dhundho (existing user_id=2 entries ke beech)
 *   Step 4: Index mein naya entry daalo
 *   Step 5: B-Tree rebalance karo agar zaroorat ho
 *
 *   Ek INSERT = Table write + Index write (2 writes!)
 *
 * COST 3: Index Creation Time
 *   → Existing large table pe index banane mein time lagta hai.
 *   → 10 lakh rows ke liye 1-2 minutes lag sakte hain.
 *
 * ✅ Index BANAO jab:
 *   → Query bahut baar chalti hai (din mein 100+ baar)
 *   → Query bina index ke slow hai
 *   → Table badi hai (10,000+ rows)
 *
 * ❌ Index MAT BANAO jab:
 *   → Query kabhi kabhi chalti hai (mahine mein ek baar)
 *   → Table mein bahut zyada INSERT/UPDATE hote hain
 *   → Table pe pehle se 5-6 indexes hain
 *   → Column mein bahut kam unique values hain (jaise status with 4 values)
 */


-- Index space check karo
SELECT
    indexname,
    pg_size_pretty(pg_indexes_size('orders'::regclass)) AS index_size
FROM pg_indexes
WHERE tablename = 'orders';

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-2: QUERY OPTIMIZATION
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * 2.1 DO NOT USE SELECT * — Sirf Zaroorat Ke Columns Lo
 * ───────────────────────────────────────────────────────
 *   SELECT * matlab "mujhe saare columns do."
 *   Ye time aur memory waste karta hai.
 *   Sirf wahi columns lo jo tumhe chahiye.
 */


-- ❌ BAD: Sab columns (unnecessary data bhi aata hai)
SELECT * FROM orders WHERE user_id = 1;

-- ✅ GOOD: Sirf zaroori columns
SELECT order_id, product_name, price FROM orders WHERE user_id = 1;


/**
 * WHY?
 *   → Kam data DB se app tak travel karta hai
 *   → Kam memory use hoti hai
 *   → Query faster chalti hai
 */

In [ ]:
/**
 * 2.2 SARGABLE CONDITIONS — Index-Friendly WHERE Clause
 * ──────────────────────────────────────────────────────
 * Hinglish:
 *   SARGABLE = aisi conditions jo index use kar sakein.
 *   WHERE clause aise likho ki index kaam aaye.
 *
 *   RULE: Column pe function ya calculation mat lagao.
 *   Calculation doosri side (value side) pe karo.
 */


-- ❌ BAD: Column pe function (index nahi chalega)
SELECT * FROM orders WHERE DATE(order_date) = '2024-01-15';

-- ✅ GOOD: Direct comparison (index chalega)
SELECT * FROM orders WHERE order_date = '2024-01-15';


-- ❌ BAD: Column pe calculation
SELECT * FROM orders WHERE price * 1.1 > 1000;

-- ✅ GOOD: Calculation doosri side pe karo
SELECT * FROM orders WHERE price > 1000 / 1.1;


-- ❌ BAD: LIKE with % pehle (index nahi chalega)
SELECT * FROM orders WHERE product_name LIKE '%lap%';

-- ✅ GOOD: LIKE with % baad mein (index chalega)
SELECT * FROM orders WHERE product_name LIKE 'lap%';

In [ ]:
/**
 * 2.3 WHAT IS CACHING? — Baar Baar Use Hone Wala Data Memory Mein
 * ─────────────────────────────────────────────────────────────────
 *   Caching = frequently use hone wala data memory mein rakhna fast access ke liye.
 *
 *   Real-life example:
 *     Phone ke contacts memory mein save hote hain.
 *     Har baar phonebook khol ke dekhna nahi padta.
 *
 *   Database caching:
 *     DB automatically popular data RAM mein rakhta hai.
 *     Frequently used index pages cache ho jaate hain.
 *     Same query 2nd baar run karo → 1st se fast hogi (cached data se).
 *
 *   Better caching ke tips:
 *     → Same query same parameters ke saath run karo
 *     → Ek baar mein bahut zyada data load mat karo
 *     → LIMIT use karo — sirf zaroori rows fetch karo
 */

In [ ]:
/**
 * 2.4 USE EXPLAIN — Query Plan Check Karo
 * ─────────────────────────────────────────
 * Hinglish:
 *   EXPLAIN batata hai ki PostgreSQL tumhari query kaise run karega.
 *   Production mein dalne se pehle HAMESHA EXPLAIN check karo.
 *
 *   Output mein kya dekhna hai:
 *     Seq Scan        → BAD  (poori table scan kar raha hai)
 *     Index Scan      → GOOD (index use kar raha hai)
 *     Index Only Scan → BEST (sirf index, table access nahi)
 */


-- Basic EXPLAIN (plan dikhata hai, actual time nahi)
EXPLAIN SELECT * FROM orders WHERE user_id = 1;

-- Detailed EXPLAIN (actual time bhi dikhata hai)
EXPLAIN ANALYZE SELECT * FROM orders WHERE user_id = 1;


/**
 * GOOD output example:
 *   Index Scan using idx_orders_user_id on orders
 *   (cost=0.42..8.44 rows=10 width=48)
 *   → "Index Scan" = index use ho raha hai ✅
 *
 * BAD output example:
 *   Seq Scan on orders
 *   (cost=0.00..500.00 rows=1000 width=48)
 *   → "Seq Scan" = koi index nahi, sab rows scan ❌
 *   → Iska matlab: index add karo!
 */

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-3: LIMIT AND BATCHING
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * 3.1 USE LIMIT — Sirf Zaroorat Ki Rows Lo
 * ──────────────────────────────────────────
 *   LIMIT restrict karta hai kitne rows return honge.
 *   Jab sab rows ki zaroorat nahi — HAMESHA LIMIT use karo.
 */


-- ❌ BAD: Sab rows fetch karo (badi table mein bahut slow)
SELECT * FROM orders ORDER BY order_date DESC;

-- ✅ GOOD: Sirf 10 rows
SELECT * FROM orders ORDER BY order_date DESC LIMIT 10;


/**
 * WHY LIMIT?
 *   → Kam data transfer
 *   → Kam memory use
 *   → Faster response time
 */

In [ ]:
/**
 * 3.2 BATCHING — Bade Operations Chhote Parts Mein Karo
 * ───────────────────────────────────────────────────────
 *   Batching = bada kaam chhote chhote tukdon mein karna.
 *   Lakhon rows ek saath update karo → poori table lock ho jaayegi.
 *   100-100 rows batches mein karo → table free rehti hai.
 */


-- ❌ BAD: Ek saath sab rows update karo (table lock!)
UPDATE orders
SET status = 'OLD'
WHERE order_date < '2024-01-01';

-- ✅ GOOD: 100-100 rows ke batches mein karo
DO $$
DECLARE
    batch_size INT := 100;
BEGIN
    LOOP
        UPDATE orders
        SET status = 'OLD'
        WHERE order_id IN (
            SELECT order_id FROM orders
            WHERE order_date < '2024-01-01'
            LIMIT batch_size
        );

        EXIT WHEN NOT FOUND;
        COMMIT;
    END LOOP;
END $$;


/**
 * BATCHING ke fayde:
 *   → Table zyada der lock nahi hoti
 *   → Beech mein doosri queries chal sakti hain
 *   → Agar batch fail ho toh sirf wo batch retry karo
 */

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-4: PAGINATION — OFFSET vs CURSOR
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * 4.1 OFFSET PAGINATION — Slow Wala Tarika
 * ──────────────────────────────────────────
 *   OFFSET matlab "pehle N rows skip karo, phir next rows do."
 *   Page number badhne ke saath query BAHUT slow hoti jaati hai.
 *   Kyunki DB ko pehle sab skipped rows bhi scan karni padti hain.
 */


-- Page 1: Pehle 10 rows (0 skip)
SELECT * FROM orders ORDER BY order_id LIMIT 10 OFFSET 0;

-- Page 2: Next 10 rows (pehle 10 skip)
SELECT * FROM orders ORDER BY order_id LIMIT 10 OFFSET 10;

-- Page 3: Next 10 rows (pehle 20 skip)
SELECT * FROM orders ORDER BY order_id LIMIT 10 OFFSET 20;


/**
 * OFFSET ka problem:
 *   Page 1000 ke liye (OFFSET 10000):
 *     DB kya karta hai:
 *       1. Pehle 10,000 rows scan karta hai
 *       2. Sab skip karta hai
 *       3. Phir next 10 return karta hai
 *
 *     10,010 rows scan ki lekin return sirf 10!
 *
 *   Page 1:    0.01 seconds
 *   Page 100:  0.5 seconds
 *   Page 1000: 5 seconds  (bahut slow!)
 *
 *   Jaise page number badhta hai → query aur slow hoti hai.
 */

In [ ]:
/**
 * 4.2 CURSOR PAGINATION — Fast Wala Tarika
 * ──────────────────────────────────────────
 *   CURSOR OFFSET ki jagah WHERE condition use karta hai.
 *   "Is ID ke baad wali rows do" — seedha wahan se shuru karo.
 *   Har page SAME speed se load hota hai — chahe page 1 ho ya page 10,000.
 */


-- Page 1: Pehle 10 rows
SELECT * FROM orders ORDER BY order_id LIMIT 10;
-- Last order_id yaad rakho → maano 10 hai

-- Page 2: order_id > 10 wali rows
SELECT * FROM orders
WHERE order_id > 10
ORDER BY order_id
LIMIT 10;
-- Last order_id yaad rakho → maano 20 hai

-- Page 3: order_id > 20 wali rows
SELECT * FROM orders
WHERE order_id > 20
ORDER BY order_id
LIMIT 10;


/**
 * CURSOR ka fayda:
 *   Har page same time leta hai!
 *     Page 1:     0.01 seconds
 *     Page 100:   0.01 seconds
 *     Page 1000:  0.01 seconds  ← same speed!
 *
 * Kab konsa use karo:
 * ┌──────────────────────────┬──────────────────────────────────────────┐
 * │ OFFSET Pagination        │ CURSOR Pagination                        │
 * ├──────────────────────────┼──────────────────────────────────────────┤
 * │ Chhoti tables (few 1000s)│ Badi tables (lakhon rows)                │
 * │ Direct page jump chahiye │ Infinite scroll (Instagram, Twitter)     │
 * │ Admin panels             │ Mobile apps                              │
 * │                          │ APIs                                     │
 * └──────────────────────────┴──────────────────────────────────────────┘
 */

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-5: GOLDEN RULES — Important Points to Remember
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * DATABASE OPTIMIZATION ke Core Principles
 * ──────────────────────────────────────────
 *   Ye 10 rules yaad rakho — production DB optimize karne ke liye.
 *
 *  1. ✅ INDEX SELECT FAST KARTA HAI, INSERT/UPDATE SLOW KARTA HAI
 *        Reading ke liye acha, writing ke liye cost hai.
 *
 *  2. ✅ TABLE SCAN = BURA (SEQ SCAN)
 *        Agar EXPLAIN mein "Seq Scan" dikhta hai → index add karo.
 *
 *  3. ✅ INDEX SCAN = ACHA
 *        EXPLAIN mein "Index Scan" → tum sahi track pe ho.
 *
 *  4. ✅ COMPOSITE INDEX MEIN COLUMN ORDER MATTER KARTA HAI
 *        Sabse zyada selective column pehle rakho (most unique values).
 *
 *  5. ✅ KABHI SELECT * USE MAT KARO
 *        Sirf wahi columns lo jo tumhe chahiye.
 *
 *  6. ✅ WHERE CLAUSE INDEX-FRIENDLY BANAO (SARGABLE)
 *        Column pe function ya calculation mat lagao.
 *        Calculation value side pe karo.
 *
 *  7. ✅ HAMESHA EXPLAIN USE KARO
 *        Production mein jaane se pehle check karo ki index use ho raha hai.
 *
 *  8. ✅ BADI TABLES KE LIYE CURSOR PAGINATION USE KARO
 *        OFFSET slow hota hai bade pages mein, CURSOR hamesha fast.
 *
 *  9. ✅ BADE UPDATES BATCHING MEIN KARO
 *        Ek baar mein 10 lakh rows update mat karo.
 *
 * 10. ✅ UNUSED INDEXES HATAO
 *        Sirf writes slow karte hain. Koi fayda nahi.
 */


-- Unused indexes dhundho PostgreSQL mein
SELECT
    schemaname,
    tablename,
    indexname,
    idx_scan AS times_index_used
FROM pg_stat_user_indexes
WHERE idx_scan = 0
ORDER BY tablename;

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-6: PRACTICE EXERCISES — Try Yourself
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * EXERCISE 1: Index Banao
 * ────────────────────────
 * Query: SELECT * FROM products WHERE category = 'Electronics' AND price > 1000;
 * Konsa index banoge?
 *
 * Answer:
 *   CREATE INDEX idx_category_price ON products(category, price);
 *   category (equality) pehle, price (range) baad mein.
 *
 * EXERCISE 2: Composite Index Order
 * ────────────────────────────────────
 * Query: SELECT * FROM orders WHERE status = 'DELIVERED' AND user_id = 500;
 * Konsa better hai?
 *   A: (status, user_id)
 *   B: (user_id, status)
 *
 * Answer: B — user_id mein zyada unique values hain → pehle rakho.
 *
 * EXERCISE 3: Problem Dhundho
 * ─────────────────────────────
 * Query: SELECT * FROM users WHERE DATE(created_at) = '2024-01-01';
 * Kya problem hai?
 *
 * Answer: DATE() function column pe hai → index use nahi hoga.
 * Fix:    SELECT * FROM users WHERE created_at = '2024-01-01';
 *
 * EXERCISE 4: Index Lagana Chahiye?
 * ────────────────────────────────────
 * Column: is_active (values: TRUE ya FALSE, 50% each)
 * Index lagana chahiye?
 *
 * Answer: NO — low selectivity — index koi khaas help nahi karega.
 *         DB table scan se hi fast hoga kyunki 50% rows match karte hain.
 *
 * EXERCISE 5: Konsa Pagination Use Karo?
 * ────────────────────────────────────────
 * Mobile app with infinite scroll, 10 million rows
 * OFFSET ya CURSOR?
 *
 * Answer: CURSOR pagination — OFFSET 10 million rows pe bahut slow hoga.
 *
 * EXERCISE 6: Index Size Check Karo
 * ─────────────────────────────────────
 * orders table ke indexes kitna space le rahe hain?
 *
 * Answer:
 *   SELECT pg_size_pretty(pg_indexes_size('orders'));
 *
 * EXERCISE 7: Index Use Ho Raha Hai?
 * ─────────────────────────────────────
 * EXPLAIN se check karo ki query index use kar rahi hai ya nahi.
 *
 * Answer:
 *   EXPLAIN SELECT user_id FROM orders WHERE user_id = 1;
 *   Output mein "Index Only Scan" dhundho.
 */

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * Part-7: QUICK REFERENCE CARD
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * IMPORTANT QUESTIONS & ANSWERS:
 * ┌─────────────────────────────┬────────────────────────────────────────────────┐
 * │ Sawaal                      │ Jawab                                          │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Index kahan store hota hai? │ DISK pe (hard drive) + RAM mein cache          │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Index structure kya hai?    │ B-Tree (Balanced Tree)                         │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Index kya store karta hai?  │ 1. Column value (key) + 2. Row pointer (CTID)  │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Index search kitna fast?    │ O(log n) → 10 lakh rows ke liye sirf ~20 steps │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Index kahan lagao?          │ WHERE, JOIN, ORDER BY, GROUP BY columns        │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Index kahan mat lagao?      │ Low selectivity, rarely used, write-heavy      │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ Composite index order?      │ Sabse selective column PEHLE                   │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ SELECT *                    │ KABHI MAT use karo (sirf zaroori columns)      │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ EXPLAIN kab use karo?       │ HAMESHA — production mein daalne se pehle      │
 * ├─────────────────────────────┼────────────────────────────────────────────────┤
 * │ OFFSET vs CURSOR?           │ Chhoti table: OFFSET | Badi table: CURSOR      │
 * └─────────────────────────────┴────────────────────────────────────────────────┘
 *
 * EXPLAIN OUTPUT KYA MATLAB:
 * ┌──────────────────────┬─────────────────────────────────────────────────┐
 * │ EXPLAIN Output       │ Matlab                                          │
 * ├──────────────────────┼─────────────────────────────────────────────────┤
 * │ Seq Scan             │ BAD — poori table scan, koi index nahi          │
 * │ Index Scan           │ GOOD — index use ho raha hai                    │
 * │ Index Only Scan      │ BEST — sirf index, table access bilkul nahi     │
 * └──────────────────────┴─────────────────────────────────────────────────┘
 *
 * CLEANUP COMMANDS:
 *
 *   DROP INDEX IF EXISTS idx_user_id;
 *   DROP INDEX IF EXISTS idx_user_status;
 *   DROP INDEX IF EXISTS idx_covering;
 *   DROP INDEX IF EXISTS idx_user_orderdate;
 *   DROP INDEX IF EXISTS idx_order_date;
 *   DROP INDEX IF EXISTS idx_orders_user_id;
 *   DROP TABLE IF EXISTS orders;
 *   DROP TABLE IF EXISTS users;
 */

In [ ]:
/**
 * ──────────────────────────────────────────────────────────────────────
 * FINAL SUMMARY — Ye sab tumne seekha
 * ──────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 *  1. INDEX = Phonebook jo DB ko data fast dhundhne deta hai
 *
 *  2. INDEX STORAGE = DISK pe store hota hai + frequently used parts RAM mein cache
 *
 *  3. B-TREE = Technology jo index fast banati hai (log n time)
 *     → 10 lakh rows → sirf 20 steps
 *
 *  4. INDEX DATA = Column value + row pointer (CTID) — full row nahi
 *
 *  5. SPEED = Index = 200x faster than table scan for large tables
 *
 *  6. TABLE SCAN = BAD (Seq Scan — sab rows check)
 *     INDEX SCAN = GOOD (seedha matching rows pe jaata hai)
 *
 *  7. COMPOSITE INDEX = Multiple columns pe index
 *     → Column order matters! Equality first, range second
 *
 *  8. SELECT * = KABHI USE MAT KARO
 *     → Sirf zaroori columns select karo
 *
 *  9. SARGABLE = WHERE clause aise likho ki index use ho sake
 *     → Column pe function mat lagao
 *
 * 10. EXPLAIN = HAMESHA use karo query check karne ke liye
 *
 * 11. CURSOR PAGINATION = Badi tables ke liye
 *     → OFFSET slow hota jaata hai bade pages mein
 *
 * 12. BATCHING = Bade updates chhote parts mein karo
 *
 * 13. UNUSED INDEXES HATAO = Ye sirf writes slow karte hain
 *
 * REMEMBER:
 *   Ek well-optimized database 1000x faster ho sakta hai poorly optimized se!
 *   Production mein daalne se pehle HAMESHA EXPLAIN se check karo.
 *   Indexes powerful tools hain — lekin inki cost hai. Sochke use karo!
 */